In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PREPROCESSING CROSS-WEEK - w2 & w3 (Version finale adaptée)
# ═══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

CHEMIN_w2 = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_new_w3\Logistics_Preprocessed_Final.xlsx"
CHEMIN_W3 = r"C:\Users\INFOTEC\OneDrive\Bureau\data_lake_w3\Input_Logistic_w3.xlsx"
DOSSIER_SORTIE = r"C:\Users\INFOTEC\OneDrive\Bureau\Pre_w3w3\Cross_Week_Results"

os.makedirs(DOSSIER_SORTIE, exist_ok=True)

CHEMIN_SORTIE_w2 = os.path.join(DOSSIER_SORTIE, 'Week2_Final.xlsx')
CHEMIN_SORTIE_w3 = os.path.join(DOSSIER_SORTIE, 'Week3_Final.xlsx')

# ═══════════════════════════════════════════════════════════════════════════════
# FONCTIONS DE CONVERSION
# ═══════════════════════════════════════════════════════════════════════════════

def virgule_vers_nombre(valeur):
    """Convertit une valeur avec virgule en nombre"""
    if pd.isna(valeur):
        return np.nan
    if isinstance(valeur, (int, float)):
        return float(valeur)
    valeur_str = str(valeur).strip().replace(' ', '').replace(',', '.')
    if valeur_str in ['', 'nan', 'NaN', 'None', '-']:
        return np.nan
    try:
        return float(valeur_str)
    except:
        return np.nan

def nombre_vers_virgule(valeur, decimales=2):
    """Convertit un nombre en string avec virgule"""
    if pd.isna(valeur):
        return ''
    try:
        return f"{float(valeur):.{decimales}f}".replace('.', ',')
    except:
        return ''

# ═══════════════════════════════════════════════════════════════════════════════
# FONCTIONS DE PREPROCESSING
# ═══════════════════════════════════════════════════════════════════════════════

def convertir_colonnes_numeriques(df):
    """Convertit toutes les colonnes numériques de virgule vers point"""
    colonnes_numeriques = [
        'Real Inventory (Qty)', 'Stock Value (€)', 'Unit Price (€)',
        'Weekly Usage (Qty)', 'Weekly Target', 'Gap', 'Gap +', 'Gap -',
        'WU', 'Obs', 'TT Cons-', '#'
    ]
    for col in colonnes_numeriques:
        if col in df.columns:
            df[col] = df[col].apply(virgule_vers_nombre)
    return df

def appliquer_formule_stock(df):
    """Applique et corrige les formules: Stock Value = Unit Price × Real Inventory"""
    if not all(col in df.columns for col in ['Unit Price (€)', 'Real Inventory (Qty)', 'Stock Value (€)']):
        return df, {}
    
    compteurs = {'stock_value': 0, 'unit_price': 0, 'real_inventory': 0, 'incoherence': 0}
    
    has_price = df['Unit Price (€)'].notna() & (df['Unit Price (€)'] > 0)
    has_qty = df['Real Inventory (Qty)'].notna() & (df['Real Inventory (Qty)'] >= 0)
    has_value = df['Stock Value (€)'].notna() & (df['Stock Value (€)'] > 0)
    
    # Calculer Stock Value si manquant
    mask = (~has_value) & has_price & has_qty
    if mask.any():
        df.loc[mask, 'Stock Value (€)'] = df.loc[mask, 'Unit Price (€)'] * df.loc[mask, 'Real Inventory (Qty)']
        compteurs['stock_value'] = mask.sum()
    
    # Calculer Unit Price si manquant
    mask = (~has_price) & has_value & has_qty
    if mask.any():
        df.loc[mask, 'Unit Price (€)'] = df.loc[mask, 'Stock Value (€)'] / df.loc[mask, 'Real Inventory (Qty)']
        compteurs['unit_price'] = mask.sum()
    
    # Calculer Real Inventory si manquant
    mask = (~has_qty) & has_value & has_price
    if mask.any():
        df.loc[mask, 'Real Inventory (Qty)'] = df.loc[mask, 'Stock Value (€)'] / df.loc[mask, 'Unit Price (€)']
        compteurs['real_inventory'] = mask.sum()
    
    # Corriger les incohérences (>10% différence)
    mask = has_price & has_qty & has_value
    if mask.any():
        stock_theorique = df.loc[mask, 'Unit Price (€)'] * df.loc[mask, 'Real Inventory (Qty)']
        stock_actuel = df.loc[mask, 'Stock Value (€)']
        mask_incoherent = mask & (abs(stock_actuel - stock_theorique) / (stock_actuel + 0.01) > 0.10)
        if mask_incoherent.any():
            df.loc[mask_incoherent, 'Stock Value (€)'] = stock_theorique[mask_incoherent]
            compteurs['incoherence'] = mask_incoherent.sum()
    
    # Si Real Inventory = 0, alors Stock Value = 0
    mask = (df['Real Inventory (Qty)'].fillna(0) == 0)
    df.loc[mask, 'Stock Value (€)'] = 0
    
    return df, compteurs

def imputer_valeurs(df):
    """Impute les valeurs manquantes avec des stratégies appropriées"""
    # Real Inventory: remplir par 0
    if 'Real Inventory (Qty)' in df.columns:
        df['Real Inventory (Qty)'] = df['Real Inventory (Qty)'].fillna(0)
        df.loc[df['Real Inventory (Qty)'] < 0, 'Real Inventory (Qty)'] = 0
    
    # Prix et valeur: remplir par 0
    for col in ['Unit Price (€)', 'Stock Value (€)']:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    # Weekly Usage et Target: médiane
    for col in ['Weekly Usage (Qty)', 'Weekly Target']:
        if col in df.columns:
            med = df[col].median()
            df[col] = df[col].fillna(med if pd.notna(med) else 0)
    
    # Gap et autres: moyenne
    for col in ['Gap', 'Gap +', 'Gap -', 'WU', 'Obs']:
        if col in df.columns:
            moy = df[col].mean()
            df[col] = df[col].fillna(moy if pd.notna(moy) else 0)
    
    # TT Cons- et #: 0
    for col in ['TT Cons-', '#']:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    # Colonnes textuelles
    for col in ['Supplier', 'Cons']:
        if col in df.columns:
            mode = df[col].mode()
            df[col] = df[col].fillna(mode[0] if len(mode) > 0 else 'Unknown')
    
    # Description depuis Part Number
    if 'Description' in df.columns and 'Part Number' in df.columns:
        df['Description'] = df['Description'].fillna(df['Part Number'].astype(str))
    
    return df

def reconvertir_virgules(df):
    """Reconvertit les colonnes numériques vers format virgule"""
    colonnes_numeriques = [
        'Real Inventory (Qty)', 'Stock Value (€)', 'Unit Price (€)',
        'Weekly Usage (Qty)', 'Weekly Target', 'Gap', 'Gap +', 'Gap -',
        'WU', 'Obs', 'TT Cons-', '#'
    ]
    for col in colonnes_numeriques:
        if col in df.columns:
            df[col] = df[col].apply(lambda x: nombre_vers_virgule(x) if pd.notna(x) else '')
    return df

# ═══════════════════════════════════════════════════════════════════════════════
# GESTION PRODUITS MANQUANTS
# ═══════════════════════════════════════════════════════════════════════════════

def gerer_produits_manquants(df_w2, df_w3, sheet_name):
    """
    Gère les produits présents dans w2 mais pas w3, et vice-versa.
    
    RÈGLES:
    - Produit en w2 uniquement → Consommé totalement pendant w2
      • Ajouté à w3 avec Real Inventory = 0, Stock Value = 0
      • Weekly Usage w2 = Real Inventory w2 (consommation totale)
      
    - Produit en w3 uniquement → Nouveau produit arrivé après w2
      • Ajouté à w2 avec Real Inventory = 0, Stock Value = 0
      • Weekly Usage w2 = 0, Weekly Target w2 = 0
    """
    
    print(f"\n   📦 Gestion produits manquants - {sheet_name}")
    
    # Identifier les produits manquants
    parts_w2 = set(df_w2['Part Number'].unique())
    parts_w3 = set(df_w3['Part Number'].unique())
    
    only_in_w2 = parts_w2 - parts_w3
    only_in_w3 = parts_w3 - parts_w2
    
    print(f"      • Produits uniquement en w2: {len(only_in_w2)}")
    print(f"      • Produits uniquement en w3: {len(only_in_w3)}")
    
    # ─────────────────────────────────────────────────────────────────────────
    # Cas 1: Produits en w2 mais pas en w3 (consommés complètement)
    # ─────────────────────────────────────────────────────────────────────────
    
    if len(only_in_w2) > 0:
        for part_num in only_in_w2:
            # Récupérer données w2
            row_w2 = df_w2[df_w2['Part Number'] == part_num].iloc[0]
            
            # Mettre à jour Weekly Usage w2 = Real Inventory w2 (consommé tout)
            real_inv_w2 = row_w2['Real Inventory (Qty)']
            df_w2.loc[df_w2['Part Number'] == part_num, 'Weekly Usage (Qty)'] = real_inv_w2
            
            # Créer ligne pour w3 avec inventaire = 0
            new_row_w3 = {}
            
            # Copier colonnes depuis w2
            cols_to_copy = ['Part Number', 'Description', 'Supplier', 'Cons', 'Unit Price (€)', 'pays']
            for col in cols_to_copy:
                if col in df_w3.columns and col in row_w2.index:
                    new_row_w3[col] = row_w2[col]
            
            # Mettre à 0: inventaire et valeur
            cols_zero = ['Real Inventory (Qty)', 'Stock Value (€)']
            for col in cols_zero:
                if col in df_w3.columns:
                    new_row_w3[col] = 0
            
            # Initialiser autres colonnes numériques
            other_cols = ['Weekly Usage (Qty)', 'Weekly Target', 'Gap', 'Gap +', 'Gap -', 
                         'WU', 'Obs', 'TT Cons-', '#']
            for col in other_cols:
                if col in df_w3.columns:
                    new_row_w3[col] = 0
            
            # Ajouter à w3
            df_w3 = pd.concat([df_w3, pd.DataFrame([new_row_w3])], ignore_index=True)
        
        print(f"      ✅ {len(only_in_w2)} produits ajoutés à w3 (consommés en w2)")
    
    # ─────────────────────────────────────────────────────────────────────────
    # Cas 2: Produits en w3 mais pas en w2 (nouveaux produits)
    # ─────────────────────────────────────────────────────────────────────────
    
    if len(only_in_w3) > 0:
        for part_num in only_in_w3:
            # Récupérer données w3
            row_w3 = df_w3[df_w3['Part Number'] == part_num].iloc[0]
            
            # Créer ligne pour w2 avec inventaire = 0
            new_row_w2 = {}
            
            # Copier colonnes depuis w3
            cols_to_copy = ['Part Number', 'Description', 'Supplier', 'Cons', 'Unit Price (€)', 'pays']
            for col in cols_to_copy:
                if col in df_w2.columns and col in row_w3.index:
                    new_row_w2[col] = row_w3[col]
            
            # Mettre à 0: inventaire, valeur, target
            cols_zero = ['Real Inventory (Qty)', 'Stock Value (€)', 'Weekly Target', 'Weekly Usage (Qty)']
            for col in cols_zero:
                if col in df_w2.columns:
                    new_row_w2[col] = 0
            
            # Initialiser autres colonnes numériques
            other_cols = ['Gap', 'Gap +', 'Gap -', 'WU', 'Obs', 'TT Cons-', '#']
            for col in other_cols:
                if col in df_w2.columns:
                    new_row_w2[col] = 0
            
            # Ajouter à w2
            df_w2 = pd.concat([df_w2, pd.DataFrame([new_row_w2])], ignore_index=True)
        
        print(f"      ✅ {len(only_in_w3)} produits ajoutés à w2 (nouveaux en w3)")
    
    return df_w2, df_w3

# ═══════════════════════════════════════════════════════════════════════════════
# FONCTION PRINCIPALE
# ═══════════════════════════════════════════════════════════════════════════════

def preprocessing_cross_week():
    """Fonction principale de preprocessing cross-week"""
    
    print("\n" + "="*80)
    print("🚀 PREPROCESSING CROSS-WEEK - w2 & w3")
    print("="*80)

    try:
        w2_file = pd.ExcelFile(CHEMIN_w2)
        w3_file = pd.ExcelFile(CHEMIN_W3)
    except Exception as e:
        print(f"❌ ERREUR lors de l'ouverture des fichiers: {str(e)}")
        return

    w2_results = {}
    w3_results = {}

    for sheet_name in w2_file.sheet_names:
        if sheet_name not in w3_file.sheet_names:
            print(f"\n⚠️  Sheet '{sheet_name}' existe en w2 mais pas en w3 - Ignoré")
            continue
        
        print(f"\n{'='*80}")
        print(f"📊 TRAITEMENT: {sheet_name}")
        print(f"{'='*80}")
        
        # Charger les données
        df_w2 = pd.read_excel(CHEMIN_w2, sheet_name=sheet_name)
        df_w3 = pd.read_excel(CHEMIN_w3, sheet_name=sheet_name)
        
        print(f"   📥 Lignes initiales - w2: {len(df_w2)} | w3: {len(df_w3)}")

        # Convertir colonnes numériques
        df_w2 = convertir_colonnes_numeriques(df_w2)
        df_w3 = convertir_colonnes_numeriques(df_w3)

        # Nettoyer Part Number
        for df in [df_w2, df_w3]:
            if 'Part Number' in df.columns:
                df['Part Number'] = df['Part Number'].astype(str).str.strip()

        # ═══════════════════════════════════════════════════════════════════════
        # GESTION PRODUITS MANQUANTS
        # ═══════════════════════════════════════════════════════════════════════
        
        df_w2, df_w3 = gerer_produits_manquants(df_w2, df_w3, sheet_name)

        # ═══════════════════════════════════════════════════════════════════════
        # MERGE ET CALCUL WEEKLY USAGE
        # ═══════════════════════════════════════════════════════════════════════

        print(f"\n   🔄 Merge des données")
        
        # Colonnes pour merge
        cols_w2 = [c for c in ['Part Number', 'Real Inventory (Qty)', 'Unit Price (€)', 
                   'Stock Value (€)', 'Weekly Usage (Qty)', 'Description', 'Supplier', 
                   'Cons', 'pays'] if c in df_w2.columns]
        cols_w3 = [c for c in ['Part Number', 'Real Inventory (Qty)', 'Unit Price (€)', 
                   'Stock Value (€)'] if c in df_w3.columns]

        merge_df = df_w2[cols_w2].merge(
            df_w3[cols_w3],
            on='Part Number',
            how='outer',
            suffixes=('_w2', '_w3')
        )

        # Gérer doublons (garder première occurrence)
        merge_df = merge_df.groupby('Part Number').first().reset_index()

        # ═══════════════════════════════════════════════════════════════════════
        # CALCUL WEEKLY USAGE = Différence positive w2 - w3
        # ═══════════════════════════════════════════════════════════════════════
        
        print(f"\n   🔢 Calcul Weekly Usage (w2 - w3)")
        
        if 'Real Inventory (Qty)_w2' in merge_df.columns and 'Real Inventory (Qty)_w3' in merge_df.columns:
            # Calculer différence
            merge_df['Weekly Usage Calculated'] = (
                merge_df['Real Inventory (Qty)_w2'].fillna(0) - 
                merge_df['Real Inventory (Qty)_w3'].fillna(0)
            )
            
            # Si différence négative (augmentation stock), mettre à 0
            merge_df.loc[merge_df['Weekly Usage Calculated'] < 0, 'Weekly Usage Calculated'] = 0
            
            # Remplacer Weekly Usage existant dans w2
            merge_df['Weekly Usage (Qty)_w2'] = merge_df['Weekly Usage Calculated']

            nb_calcules = (merge_df['Weekly Usage Calculated'] > 0).sum()
            total_usage = merge_df['Weekly Usage Calculated'].sum()
            
            print(f"      • Articles avec usage > 0: {nb_calcules}")
            print(f"      • Usage total calculé: {total_usage:.2f}")

        # ═══════════════════════════════════════════════════════════════════════
        # SYNCHRONISATION UNIT PRICE
        # ═══════════════════════════════════════════════════════════════════════
        
        print(f"\n   💰 Synchronisation Unit Price")
        
        if 'Unit Price (€)_w2' in merge_df.columns and 'Unit Price (€)_w3' in merge_df.columns:
            # w2 → w3
            mask = merge_df['Unit Price (€)_w3'].isna() & merge_df['Unit Price (€)_w2'].notna()
            nb_w2_to_w3 = mask.sum()
            merge_df.loc[mask, 'Unit Price (€)_w3'] = merge_df.loc[mask, 'Unit Price (€)_w2']
            
            # w3 → w2
            mask = merge_df['Unit Price (€)_w2'].isna() & merge_df['Unit Price (€)_w3'].notna()
            nb_w3_to_w2 = mask.sum()
            merge_df.loc[mask, 'Unit Price (€)_w2'] = merge_df.loc[mask, 'Unit Price (€)_w3']
            
            print(f"      • Unit Price w2→w3: {nb_w2_to_w3}")
            print(f"      • Unit Price w3→w2: {nb_w3_to_w2}")

        # ═══════════════════════════════════════════════════════════════════════
        # RECONSTRUIRE LES DATAFRAMES
        # ═══════════════════════════════════════════════════════════════════════
        
        print(f"\n   🔧 Reconstruction des DataFrames")
        
        for col in merge_df.columns:
            if col.endswith('_w2'):
                col_name = col.replace('_w2', '')
                if col_name in df_w2.columns:
                    for part_num in merge_df['Part Number']:
                        if part_num in df_w2['Part Number'].values:
                            df_w2.loc[df_w2['Part Number'] == part_num, col_name] = \
                                merge_df.loc[merge_df['Part Number'] == part_num, col].values[0]

            if col.endswith('_w3'):
                col_name = col.replace('_w3', '')
                if col_name in df_w3.columns:
                    for part_num in merge_df['Part Number']:
                        if part_num in df_w3['Part Number'].values:
                            df_w3.loc[df_w3['Part Number'] == part_num, col_name] = \
                                merge_df.loc[merge_df['Part Number'] == part_num, col].values[0]

        # ═══════════════════════════════════════════════════════════════════════
        # APPLICATION FORMULES ET IMPUTATION
        # ═══════════════════════════════════════════════════════════════════════
        
        print(f"\n   ⚙️  Application formules stock")
        df_w2, compteurs_w2 = appliquer_formule_stock(df_w2)
        df_w3, compteurs_w3 = appliquer_formule_stock(df_w3)
        
        total_w2 = sum(compteurs_w2.values())
        total_w3 = sum(compteurs_w3.values())
        print(f"      • w2: {total_w2} corrections")
        print(f"      • w3: {total_w3} corrections")

        print(f"\n   📝 Imputation valeurs manquantes")
        df_w2 = imputer_valeurs(df_w2)
        df_w3 = imputer_valeurs(df_w3)

        # Reconversion virgules
        df_w2 = reconvertir_virgules(df_w2)
        df_w3 = reconvertir_virgules(df_w3)

        # Sauvegarder résultats
        w2_results[sheet_name] = df_w2
        w3_results[sheet_name] = df_w3
        
        print(f"\n   ✅ {sheet_name} traité - w2: {len(df_w2)} lignes | w3: {len(df_w3)} lignes")

    # ═══════════════════════════════════════════════════════════════════════════
    # SAUVEGARDE FINALE
    # ═══════════════════════════════════════════════════════════════════════════
    
    print(f"\n{'='*80}")
    print(f"💾 SAUVEGARDE DES FICHIERS FINAUX")
    print(f"{'='*80}")
    
    with pd.ExcelWriter(CHEMIN_SORTIE_w2, engine='openpyxl') as writer:
        for nom, df in w2_results.items():
            df.to_excel(writer, sheet_name=nom, index=False)
            print(f"   ✅ w2 - {nom}: {len(df)} lignes")
    
    with pd.ExcelWriter(CHEMIN_SORTIE_w3, engine='openpyxl') as writer:
        for nom, df in w3_results.items():
            df.to_excel(writer, sheet_name=nom, index=False)
            print(f"   ✅ w3 - {nom}: {len(df)} lignes")
    
    print(f"\n{'='*80}")
    print(f"✅ PREPROCESSING TERMINÉ AVEC SUCCÈS!")
    print(f"{'='*80}")
    print(f"\n📁 Fichiers générés:")
    print(f"   • {CHEMIN_SORTIE_w2}")
    print(f"   • {CHEMIN_SORTIE_w3}")
    print(f"\n🎯 Opérations effectuées:")
    print(f"   • Gestion produits manquants entre w2 et w3")
    print(f"   • Calcul Weekly Usage = Real Inventory w2 - Real Inventory w3")
    print(f"   • Synchronisation Unit Price bidirectionnelle")
    print(f"   • Correction formules stock (Stock Value = Unit Price × Qty)")
    print(f"   • Imputation valeurs manquantes")

if __name__ == "__main__":
    try:
        preprocessing_cross_week()
    except Exception as e:
        print(f"\n❌ ERREUR CRITIQUE: {str(e)}")
        import traceback
        traceback.print_exc()